---

# IF4061-Moview TMDb Data Preprocessing

*Pipeline v2 uses TMDB_movie_dataset_v11 with about 27k quality filtered films as primary source, joins tmdb_5000_credits for director and cast data where IDs overlap, and exports a SQLite database (`moview.db`) plus static GeoJSON files consumed by the dashboard.*

---

## Kelompok

- (Nama anggota)
- (Nama anggota)
- (Nama anggota)

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Parse & Clean**](#3)
4. [**GeoJSON Patch**](#4)
5. [**Export SQLite**](#5)


---

# Introduction <a name="1"></a>

---

## Overview

v2 replaces the 4,803 film tmdb_5000 dataset with the larger TMDB v11 dataset with about 27k quality filtered films. Director and cast data come from the tmdb_5000_credits CSV joined by movie ID. People tab charts are scoped to the about 4,800 films where credits exist.

**Quality filter applied to v11** `status == Released`, `vote_count >= 50`, `year 1950 to 2024`.
**New fields added** `month` from 1 to 12, `release_season`, and `cast` with top 5 actor names.

## Outputs

All files written to `dataset/clean/`.

| File | Contents |
|------|----------|
| `moview.db` | SQLite database with all aggregated tables |
| `manifest.json` | Year bounds and generated timestamp |
| `geo_summary.json` | Country names, continents, total films, profit, and rating for the choropleth map |
| `countries.geojson` | Natural Earth polygons with patched ISO codes |
| `continents.geojson` | Continent union polygons |

---

# Initialization <a name="2"></a>

---

In [1]:
#%pip install -q pandas numpy shapely tqdm

In [2]:
import ast
import json
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings('ignore')

In [3]:
class Settings:
    BASE_DIR    = Path('d:/Akademik/ITB/Semester-6/IF4061-VisDat/Tubes/IF4061-Moview')
    RAW_DIR     = BASE_DIR / 'dataset' / 'raw'
    OUT_DIR     = BASE_DIR / 'dataset' / 'clean'
    DB_PATH     = OUT_DIR / 'moview.db'

    V11_CSV     = RAW_DIR / 'TMDB_movie_dataset_v11.csv'
    CREDITS_CSV = RAW_DIR / 'tmdb_5000_credits.csv'
    GEOJSON     = RAW_DIR / 'ne_110m_admin_0_countries.geojson'

    # Quality filter
    MIN_VOTES   = 50
    YEAR_MIN    = 1950
    YEAR_MAX    = 2024

CFG = Settings()
CFG.OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir is', CFG.OUT_DIR)

Output dir is d:\Akademik\ITB\Semester-6\IF4061-VisDat\Tubes\IF4061-Moview\dataset\clean


---

# Parse & Clean <a name="3"></a>

---

## 3.1 Load v11 + quality filter

In [4]:
raw = pd.read_csv(CFG.V11_CSV, low_memory=False)
print(f'Raw v11 shape is {raw.shape}')

raw['year'] = pd.to_datetime(raw['release_date'], errors='coerce').dt.year
raw['month'] = pd.to_datetime(raw['release_date'], errors='coerce').dt.month

df = raw[
    (raw['status'] == 'Released') &
    (raw['vote_count'] >= CFG.MIN_VOTES) &
    (raw['year'] >= CFG.YEAR_MIN) &
    (raw['year'] <= CFG.YEAR_MAX)
].copy()

print(f'After quality filter there are {len(df)} films')

Raw v11 shape is (1432967, 24)
After quality filter there are 26925 films


## 3.2 Load credits, extract director + cast

In [5]:
credits_raw = pd.read_csv(CFG.CREDITS_CSV)
credits_raw = credits_raw.rename(columns={'movie_id': 'id'})

def parse_col(val):
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return []

def extract_director(val):
    for m in parse_col(val):
        if m.get('job') == 'Director':
            return m.get('name')
    return None

def extract_cast(val, n=5):
    members = parse_col(val)
    members.sort(key=lambda m: m.get('order', 999))
    return [m['name'] for m in members[:n] if 'name' in m]

credits_raw['director'] = credits_raw['crew'].apply(extract_director)
credits_raw['cast_list'] = credits_raw['cast'].apply(extract_cast)

credits_slim = credits_raw[['id', 'director', 'cast_list']]
overlap = set(df['id']).intersection(set(credits_slim['id']))
print(f'Credits overlap has {len(overlap)} films out of {len(df)} total')

Credits overlap has 4224 films out of 26925 total


## 3.3 Language code to name mapping

In [6]:
LANG_NAMES = {
    'af':'Afrikaans','ak':'Akan','am':'Amharic','ar':'Arabic','az':'Azerbaijani',
    'be':'Belarusian','bg':'Bulgarian','bm':'Bambara','bn':'Bengali','bo':'Tibetan',
    'bs':'Bosnian','ca':'Catalan','cs':'Czech','cy':'Welsh','da':'Danish',
    'de':'German','dz':'Dzongkha','el':'Greek','en':'English','eo':'Esperanto',
    'es':'Spanish','et':'Estonian','eu':'Basque','fa':'Persian','ff':'Fula',
    'fi':'Finnish','fr':'French','fy':'Frisian','ga':'Irish','gl':'Galician',
    'gu':'Gujarati','ha':'Hausa','he':'Hebrew','hi':'Hindi','hr':'Croatian',
    'hu':'Hungarian','hy':'Armenian','id':'Indonesian','ig':'Igbo','is':'Icelandic',
    'it':'Italian','iu':'Inuktitut','ja':'Japanese','jv':'Javanese','ka':'Georgian',
    'kk':'Kazakh','km':'Khmer','kn':'Kannada','ko':'Korean','ku':'Kurdish',
    'ky':'Kyrgyz','lb':'Luxembourgish','lo':'Lao','lt':'Lithuanian','lv':'Latvian',
    'mg':'Malagasy','mi':'Maori','mk':'Macedonian','ml':'Malayalam','mn':'Mongolian',
    'mr':'Marathi','ms':'Malay','mt':'Maltese','my':'Burmese','nb':'Norwegian',
    'ne':'Nepali','nl':'Dutch','nn':'Norwegian Nynorsk','no':'Norwegian','ny':'Chichewa',
    'om':'Oromo','or':'Odia','pa':'Punjabi','pl':'Polish','ps':'Pashto',
    'pt':'Portuguese','qu':'Quechua','ro':'Romanian','ru':'Russian','rw':'Kinyarwanda',
    'si':'Sinhala','sk':'Slovak','sl':'Slovenian','sm':'Samoan','sn':'Shona',
    'so':'Somali','sq':'Albanian','sr':'Serbian','ss':'Swati','st':'Sotho',
    'su':'Sundanese','sv':'Swedish','sw':'Swahili','ta':'Tamil','te':'Telugu',
    'tg':'Tajik','th':'Thai','ti':'Tigrinya','tk':'Turkmen','tl':'Filipino',
    'tn':'Tswana','to':'Tongan','tr':'Turkish','ts':'Tsonga','tt':'Tatar',
    'tw':'Twi','ug':'Uyghur','uk':'Ukrainian','ur':'Urdu','uz':'Uzbek',
    've':'Venda','vi':'Vietnamese','wo':'Wolof','xh':'Xhosa','yi':'Yiddish',
    'yo':'Yoruba','zh':'Chinese','zu':'Zulu','xx':'No Language','cn':'Cantonese',
}

def lang_name(code):
    if pd.isna(code): return None
    return LANG_NAMES.get(str(code).strip().lower(), str(code).upper())

print(f'Language map loaded with {len(LANG_NAMES)} entries')

Language map loaded with 120 entries


## 3.4 Parse columns

In [7]:
def split_csv(val):
    if pd.isna(val) or str(val).strip() == '':
        return []
    return [x.strip() for x in str(val).split(',') if x.strip()]

df['genres']               = df['genres'].apply(split_csv)
df['keywords']             = df['keywords'].apply(split_csv)
df['production_companies'] = df['production_companies'].apply(split_csv)
df['spoken_languages']     = df['spoken_languages'].apply(split_csv)
df['production_countries_raw'] = df['production_countries'].apply(split_csv)
df['language']             = df['original_language'].apply(lang_name)

df['primary_genre'] = df['genres'].apply(lambda g: g[0] if g else None)
print('Columns parsed.')

Columns parsed.


## 3.4 Map production country names to ISO_A2

In [8]:
with open(CFG.GEOJSON, encoding='utf-8') as f:
    _gj = json.load(f)

# Build name to ISO_A2 from GeoJSON names
name_to_iso = {}
iso_to_display_name = {}
iso_to_continent = {}
for feat in _gj['features']:
    p = feat['properties']
    iso = p.get('ISO_A2', '')
    if not iso or iso == '-99':
        iso = p.get('ISO_A2_EH', '')
    if iso and iso != '-99':
        iso_to_display_name[iso] = p.get('NAME_LONG') or p.get('NAME') or iso
        iso_to_continent[iso] = p.get('CONTINENT')
        for key in ('NAME', 'NAME_LONG', 'FORMAL_EN', 'BRK_NAME'):
            name = p.get(key, '')
            if name:
                name_to_iso[name] = iso

# Manual overrides for common v11 country name mismatches
name_to_iso.update({
    'United States of America': 'US',
    'United Kingdom':           'GB',
    'South Korea':              'KR',
    'Russia':                   'RU',
    'Czech Republic':           'CZ',
    'Iran':                     'IR',
    'Syria':                    'SY',
    'Bolivia':                  'BO',
    'Venezuela':                'VE',
    'Tanzania':                 'TZ',
    'Moldova':                  'MD',
    'Vietnam':                  'VN',
    'Hong Kong':                'HK',
    'Taiwan':                   'TW',
    'Palestine':                'PS',
    'Macedonia':                'MK',
    'Kosovo':                   'XK',
})
iso_to_display_name.update({
    'HK': 'Hong Kong',
    'RU': 'Russia',
    'TW': 'Taiwan',
    'XK': 'Kosovo',
})
iso_to_continent.update({
    'AW': 'North America', 'GP': 'North America',
    'HK': 'Asia',          'MT': 'Europe',
    'SG': 'Asia',          'TW': 'Asia',
})

def get_primary_iso(countries):
    for c in countries:
        iso = name_to_iso.get(c)
        if iso:
            return iso
    return None

df['primary_iso'] = df['production_countries_raw'].apply(get_primary_iso)
df['continent'] = df['primary_iso'].map(iso_to_continent)

matched = df['primary_iso'].notna().sum()
print(f'ISO matched {matched}/{len(df)} films ({matched/len(df)*100:.1f}%)')

ISO matched 26544/26925 films (98.6%)


## 3.5 Derived fields + credit join

In [9]:
df['budget']  = pd.to_numeric(df['budget'],  errors='coerce').replace(0, np.nan)
df['revenue'] = pd.to_numeric(df['revenue'], errors='coerce').replace(0, np.nan)
df['profit']  = df['revenue'] - df['budget']
df['decade']  = (df['year'] // 10 * 10).astype('Int64')

SEASON = {12:'Winter',1:'Winter',2:'Winter',
          3:'Spring',4:'Spring',5:'Spring',
          6:'Summer',7:'Summer',8:'Summer',
          9:'Fall',10:'Fall',11:'Fall'}
df['release_season'] = df['month'].map(SEASON)

# Join director + cast (null for non-overlapping rows)
df = df.merge(credits_slim, on='id', how='left')
df['cast_list'] = df['cast_list'].apply(lambda x: x if isinstance(x, list) else [])

print(f'Total films {len(df)}')
print(f'With director {df["director"].notna().sum()}')
print(f'With cast {df["cast_list"].apply(bool).sum()}')

Total films 26925
With director 4223
With cast 4224


## 3.6 Financial subset + profit/budget tiers

In [10]:
fin = df.dropna(subset=['budget', 'revenue']).copy()

q1 = float(fin['profit'].quantile(0.25))
q2 = float(fin['profit'].quantile(0.50))
q3 = float(fin['profit'].quantile(0.75))
print(f'Profit tiers Q1={q1/1e6:.1f}M  Q2={q2/1e6:.1f}M  Q3={q3/1e6:.1f}M')

def assign_profit_tier(profit):
    if pd.isna(profit): return None
    if profit < q1:     return 'Low'
    if profit < q2:     return 'Mid-Low'
    if profit < q3:     return 'Mid-High'
    return 'High'

def assign_budget_tier(budget):
    if pd.isna(budget):      return None
    if budget < 10_000_000:  return 'Low'
    if budget < 50_000_000:  return 'Mid'
    return 'High'

fin['profit_tier'] = fin['profit'].apply(assign_profit_tier)
fin['budget_tier'] = fin['budget'].apply(assign_budget_tier)
print(f'Financial subset has {len(fin)} films')

Profit tiers Q1=-2.5M  Q2=9.1M  Q3=52.4M
Financial subset has 7641 films


---

# GeoJSON Patch <a name="4"></a>

---

## 4.1 countries.geojson

Same patch as v1. It fixes `ISO_A2 = -99` using `ISO_A2_EH`, drops Antarctica, and drops open ocean.

In [11]:
with open(CFG.GEOJSON, encoding='utf-8') as f:
    geojson = json.load(f)

patched = []
kept = []
for feat in geojson['features']:
    p = feat['properties']
    if p.get('CONTINENT') in ('Antarctica', 'Seven seas (open ocean)'):
        continue
    if p.get('ISO_A2') == '-99' and p.get('ISO_A2_EH', '-99') != '-99':
        p['ISO_A2'] = p['ISO_A2_EH']
        patched.append(p['NAME'])
    kept.append(feat)

geojson['features'] = kept
out_path = CFG.OUT_DIR / 'countries.geojson'
out_path.write_text(json.dumps(geojson, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Kept {len(kept)} features and patched {len(patched)} values {patched}')

Kept 175 features and patched 3 values ['Norway', 'France', 'Kosovo']


## 4.2 continents.geojson

In [12]:
from shapely.geometry import shape, mapping
from shapely.ops import unary_union
from collections import defaultdict

with open(CFG.OUT_DIR / 'countries.geojson', encoding='utf-8') as f:
    _cj = json.load(f)

by_cont = defaultdict(list)
for feat in _cj['features']:
    cont = feat['properties'].get('CONTINENT')
    if cont:
        try:
            by_cont[cont].append(shape(feat['geometry']))
        except Exception:
            pass

cont_features = [
    {'type': 'Feature', 'properties': {'continent': cont}, 'geometry': mapping(unary_union(polys))}
    for cont, polys in by_cont.items()
]
cont_out_path = CFG.OUT_DIR / 'continents.geojson'
cont_out_path.write_text(
    json.dumps({'type': 'FeatureCollection', 'features': cont_features}, indent=2, ensure_ascii=False),
    encoding='utf-8',
)
print(f'continents.geojson has {len(cont_features)} continents')

continents.geojson has 6 continents


---

# Export SQLite <a name="5"></a>

---

All aggregated data is written to `moview.db` via `DataFrame.to_sql()`. KPI tile ranges are computed on demand by the API from the `kpi` table, eliminating the O(n²) pre-computation step.

## 5.1 Helper functions

In [13]:
def sanitize_value(v):
    if isinstance(v, (np.floating, float)):
        value = float(v)
        return None if not np.isfinite(value) else value
    if isinstance(v, (np.integer, int)):
        return int(v)
    if isinstance(v, dict):
        return {k2: sanitize_value(v2) for k2, v2 in v.items()}
    if isinstance(v, list):
        return [sanitize_value(i) for i in v]
    return v

def sanitize_records(records):
    return [
        {k: sanitize_value(v) for k, v in row.items()}
        for row in records
    ]

def write_json(path, data):
    path.write_text(
        json.dumps(data, indent=2, ensure_ascii=False, allow_nan=False),
        encoding='utf-8',
    )

print('Helpers loaded.')

Helpers loaded.


## 5.2 Aggregate helpers

In [14]:
METRIC_COLS = ['budget', 'revenue', 'profit', 'vote_average', 'vote_count', 'popularity']
BASE_COLS = ['scope_type', 'scope_id', 'year', 'id'] + METRIC_COLS

def add_scope_rows(source):
    base = source[source['year'].notna()].copy()
    frames = [base.assign(scope_type='world', scope_id='WORLD')]

    continent_rows = base[base['continent'].notna()].copy()
    if not continent_rows.empty:
        frames.append(continent_rows.assign(scope_type='continent', scope_id=continent_rows['continent']))

    country_rows = base[base['primary_iso'].notna()].copy()
    if not country_rows.empty:
        frames.append(country_rows.assign(scope_type='country', scope_id=country_rows['primary_iso']))

    return pd.concat(frames, ignore_index=True)

def metric_agg(frame, group_cols):
    if frame.empty:
        return []
    out = (
        frame
        .groupby(group_cols, dropna=False)
        .agg(
            film_count=('id', 'count'),
            budget_sum=('budget', 'sum'), budget_count=('budget', 'count'),
            revenue_sum=('revenue', 'sum'), revenue_count=('revenue', 'count'),
            profit_sum=('profit', 'sum'), profit_count=('profit', 'count'),
            rating_sum=('vote_average', 'sum'), rating_count=('vote_average', 'count'),
            vote_count_sum=('vote_count', 'sum'),
            popularity_sum=('popularity', 'sum'), popularity_count=('popularity', 'count'),
        )
        .reset_index()
    )
    return sanitize_records(out.to_dict('records'))

scoped = add_scope_rows(df)
financial_scoped = add_scope_rows(fin)
print(f'Scoped movie rows {len(scoped)}')

Scoped movie rows 80013


## 5.3 Core tables

In [15]:
from datetime import datetime, timezone

def avg_from_sum_count(row, sum_col, count_col):
    count = row.get(count_col, 0)
    return row.get(sum_col, 0) / count if count else None

kpi_out = metric_agg(scoped, ['scope_type', 'scope_id', 'year'])
kpi_language_out = sanitize_records(
    scoped
    .groupby(['scope_type', 'scope_id', 'year', 'language'], dropna=False)
    .agg(film_count=('id', 'count'))
    .reset_index()
    .to_dict('records')
)

country_totals = pd.DataFrame(kpi_out)
country_totals = country_totals[country_totals['scope_type'] == 'country']
country_totals = (
    country_totals
    .groupby('scope_id', as_index=False)
    .agg(
        film_count=('film_count', 'sum'),
        profit_sum=('profit_sum', 'sum'), profit_count=('profit_count', 'sum'),
        rating_sum=('rating_sum', 'sum'), rating_count=('rating_count', 'sum'),
    )
)
geo_summary_out = []
for row in country_totals.to_dict('records'):
    iso = row['scope_id']
    geo_summary_out.append({
        'iso_a2': iso,
        'name': iso_to_display_name.get(iso, iso),
        'continent': iso_to_continent.get(iso),
        'film_count': row['film_count'],
        'avg_profit': avg_from_sum_count(row, 'profit_sum', 'profit_count'),
        'avg_rating': avg_from_sum_count(row, 'rating_sum', 'rating_count'),
    })

# geo_summary stays as static JSON — consumed directly by ChoroplethMap
write_json(CFG.OUT_DIR / 'geo_summary.json', sanitize_records(geo_summary_out))

# Open (or recreate) the SQLite database
if CFG.DB_PATH.exists():
    CFG.DB_PATH.unlink()
conn = sqlite3.connect(CFG.DB_PATH)

pd.DataFrame(kpi_out).to_sql('kpi', conn, if_exists='replace', index=False)
pd.DataFrame(kpi_language_out).to_sql('kpi_language', conn, if_exists='replace', index=False)

print(f'kpi rows {len(kpi_out)}')
print(f'kpi_language rows {len(kpi_language_out)}')
print(f'geo_summary rows {len(geo_summary_out)}')

kpi rows 2367
kpi_language rows 6387
geo_summary rows 115


## 5.4 Scope aggregates

In [16]:
genre_scoped = (
    scoped
    .explode('genres')
    .rename(columns={'genres': 'genre'})
)
genre_scoped = genre_scoped[genre_scoped['genre'].notna() & (genre_scoped['genre'] != '')]
genre_agg_out = metric_agg(genre_scoped, ['scope_type', 'scope_id', 'year', 'genre'])
pd.DataFrame(genre_agg_out).to_sql('genre_agg', conn, if_exists='replace', index=False)

financial_grouped = financial_scoped.rename(columns={'primary_genre': 'genre'})
financial_agg_out = metric_agg(
    financial_grouped,
    ['scope_type', 'scope_id', 'year', 'genre', 'budget_tier', 'release_season', 'profit_tier'],
)
pd.DataFrame(financial_agg_out).to_sql('financial_agg', conn, if_exists='replace', index=False)

director_entities = (
    scoped.loc[scoped['director'].notna(), BASE_COLS + ['director']]
    .rename(columns={'director': 'name'})
    .assign(entity_type='director')
)
studio_entities = (
    scoped.loc[:, BASE_COLS + ['production_companies']]
    .explode('production_companies')
    .rename(columns={'production_companies': 'name'})
    .assign(entity_type='studio')
)
studio_entities = studio_entities[studio_entities['name'].notna() & (studio_entities['name'] != '')]
cast_entities = (
    scoped.loc[:, BASE_COLS + ['cast_list']]
    .explode('cast_list')
    .rename(columns={'cast_list': 'name'})
    .assign(entity_type='cast')
)
cast_entities = cast_entities[cast_entities['name'].notna() & (cast_entities['name'] != '')]
people_df = pd.concat([director_entities, studio_entities, cast_entities], ignore_index=True)
people_agg_out = metric_agg(people_df, ['scope_type', 'scope_id', 'year', 'entity_type', 'name'])
pd.DataFrame(people_agg_out).to_sql('people_agg', conn, if_exists='replace', index=False)

print(f'genre_agg rows {len(genre_agg_out)}')
print(f'financial_agg rows {len(financial_agg_out)}')
print(f'people_agg rows {len(people_agg_out)}')

genre_agg rows 18811
financial_agg rows 18250
people_agg rows 232890


## 5.5 Pair, keyword and movie tables

In [17]:
pair_source_cols = BASE_COLS + ['genres']
pair_source = scoped.loc[
    scoped['genres'].apply(lambda g: isinstance(g, list) and len(g) >= 2),
    pair_source_cols,
]

pair_rows = []
for row in tqdm(pair_source.itertuples(index=False), total=len(pair_source), desc='Building genre pair rows', leave=True):
    genres = sorted(set(row.genres))
    for i, genre_a in enumerate(genres):
        for genre_b in genres[i + 1:]:
            pair_rows.append({
                'scope_type': row.scope_type, 'scope_id': row.scope_id, 'year': row.year,
                'genre_a': genre_a, 'genre_b': genre_b, 'id': row.id,
                'budget': row.budget, 'revenue': row.revenue, 'profit': row.profit,
                'vote_average': row.vote_average, 'vote_count': row.vote_count,
                'popularity': row.popularity,
            })
pair_df = pd.DataFrame(pair_rows)
genre_pair_agg_out = metric_agg(pair_df, ['scope_type', 'scope_id', 'year', 'genre_a', 'genre_b'])
pd.DataFrame(genre_pair_agg_out).to_sql('genre_pair_agg', conn, if_exists='replace', index=False)

keyword_df = (
    scoped.loc[:, ['scope_type', 'scope_id', 'year', 'genres', 'keywords']]
    .explode('genres')
    .explode('keywords')
    .rename(columns={'genres': 'genre', 'keywords': 'keyword'})
)
keyword_df = keyword_df[
    keyword_df['genre'].notna() & (keyword_df['genre'] != '') &
    keyword_df['keyword'].notna() & (keyword_df['keyword'] != '')
]
keyword_agg = (
    keyword_df
    .groupby(['scope_type', 'scope_id', 'year', 'genre', 'keyword'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values(['scope_type', 'scope_id', 'year', 'genre', 'count'], ascending=[True, True, True, True, False])
)
keyword_agg_out = sanitize_records(keyword_agg.to_dict('records'))
pd.DataFrame(keyword_agg_out).to_sql('keyword_agg', conn, if_exists='replace', index=False)

top_movie_cols = [
    'scope_type', 'scope_id', 'id', 'title', 'year', 'month', 'primary_genre',
    'primary_iso', 'continent', 'budget', 'revenue', 'profit', 'vote_average',
    'vote_count', 'popularity',
]
top_movies_out = sanitize_records(scoped.loc[:, top_movie_cols].to_dict('records'))
pd.DataFrame(top_movies_out).to_sql('top_movies', conn, if_exists='replace', index=False)

print(f'genre_pair_agg rows {len(genre_pair_agg_out)}')
print(f'keyword_agg rows {len(keyword_agg_out)}')
print(f'top_movies rows {len(top_movies_out)}')

Building genre pair rows: 100%|██████████| 61112/61112 [00:00<00:00, 71812.54it/s] 


genre_pair_agg rows 46534
keyword_agg rows 1085628
top_movies rows 80013


## 5.6 Indexes

In [18]:
conn.execute('CREATE INDEX IF NOT EXISTS idx_kpi_scope ON kpi (scope_type, scope_id)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_kpi_language_scope ON kpi_language (scope_type, scope_id)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_genre_agg_scope ON genre_agg (scope_type, scope_id)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_financial_agg_scope ON financial_agg (scope_type, scope_id)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_people_scope_type ON people_agg (scope_type, scope_id, entity_type)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_keyword_scope_genre ON keyword_agg (scope_type, scope_id, genre)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_genre_pair_scope ON genre_pair_agg (scope_type, scope_id)')
conn.execute('CREATE INDEX IF NOT EXISTS idx_topmovies_scope ON top_movies (scope_type, scope_id)')
conn.commit()
conn.close()
print(f'moview.db written to {CFG.DB_PATH}')

moview.db written to d:\Akademik\ITB\Semester-6\IF4061-VisDat\Tubes\IF4061-Moview\dataset\clean\moview.db


## 5.7 Manifest

In [19]:
manifest = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'year_min': int(df['year'].min()),
    'year_max': int(df['year'].max()),
    'db': 'moview.db',
    'core_files': ['manifest.json', 'geo_summary.json', 'countries.geojson', 'continents.geojson'],
}

write_json(CFG.OUT_DIR / 'manifest.json', manifest)
print('Manifest written.')

Manifest written.


#### Insights

> All aggregated data written to `dataset/clean/moview.db`. KPI tile ranges are not pre-computed; the API derives them on demand with a single `SELECT SUM(...)` against the `kpi` table, eliminating the 226k-row pre-computation. People data scoped to about 4,800 credit joined rows; all other charts use the full about 27k filtered set.